## Dataset

In [ ]:
# Torchvision, PyTorch çerçevesindeki (framework) bir kütüphanedir (library).
# datasets hazır (MNIST, CIFAR10 vb.) veri setlerine sağlayan bir modül.
# transforms ise veriyi modüle uygun hale getirmek için kullandığımız modüldür. 
# DataLoader ise veriyi model için uygun hale getireceğimiz bir sınıftır.
from torchvision import datasets, transforms 
from torch.utils.data import DataLoader

In [ ]:
transform = transforms.Compose([ #Pipeline
    transforms.Pad(2), # Görüntü etrafına 2 piksellik padding eklenir. 
    transforms.ToTensor(), # Görüntü PyTorch tensör formatına çevrilir. 
    transforms.Normalize((0.5,), (0.5,)) # Ortalama ve standart sapma ile normalize eder.
])

train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_data = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

In [ ]:
print("nums of train:", len(train_data))
print("nums of test:", len(test_data))

In [ ]:
image, label = train_data[0]
print("Shape of img:", image.shape)  # örn: torch.Size([1, 32, 32])
print("Label:", label)
print("Type:", type(image))

In [ ]:
image[0,1,1]

In [ ]:
import matplotlib.pyplot as plt

image, label = train_data[59999]
plt.imshow(image.squeeze(), cmap='gray')
plt.title(f"Label: {label}")
plt.show()

In [ ]:
print(image.size(0))
print(image.size(1))
print(image.size(2))

## Model (LeNet-5)

In [ ]:
# Reference: https://github.com/activatedgeek/LeNet-5
import torch.nn as nn # Katmanlar, aktivasyon fonksiyonları, loss fonksiyonları.
from collections import OrderedDict # Katmanları daha verimli tanımlama için kullanılan bir sınıf import edilir.

class C1(nn.Module): ## nn.Module bir base class
    def __init__(self):
        super(C1, self).__init__()

        self.c1 = nn.Sequential(OrderedDict([
            ('c1', nn.Conv2d(1, 6, kernel_size=(5, 5))),
            ('relu1', nn.ReLU()),
            ('s1', nn.MaxPool2d(kernel_size=(2, 2), stride=2))
        ]))

    def forward(self, img):
        output = self.c1(img)
        return output


class C2(nn.Module):
    def __init__(self):
        super(C2, self).__init__()

        self.c2 = nn.Sequential(OrderedDict([
            ('c2', nn.Conv2d(6, 16, kernel_size=(5, 5))),
            ('relu2', nn.ReLU()),
            ('s2', nn.MaxPool2d(kernel_size=(2, 2), stride=2))
        ]))

    def forward(self, img):
        output = self.c2(img)
        return output


class C3(nn.Module):
    def __init__(self):
        super(C3, self).__init__()

        self.c3 = nn.Sequential(OrderedDict([
            ('c3', nn.Conv2d(16, 120, kernel_size=(5, 5))),
            ('relu3', nn.ReLU())
        ]))

    def forward(self, img):
        output = self.c3(img)
        return output


class F4(nn.Module):
    def __init__(self):
        super(F4, self).__init__()

        self.f4 = nn.Sequential(OrderedDict([
            ('f4', nn.Linear(120, 84)),
            ('relu4', nn.ReLU())
        ]))

    def forward(self, img):
        output = self.f4(img)
        return output


class F5(nn.Module):
    def __init__(self):
        super(F5, self).__init__()

        self.f5 = nn.Sequential(OrderedDict([
            ('f5', nn.Linear(84, 10)),
            ('sig5', nn.LogSoftmax(dim=-1))
        ]))

    def forward(self, img):
        output = self.f5(img)
        return output


class LeNet5(nn.Module):
    """
    Input - 1x32x32
    Output - 10
    """
    def __init__(self):
        super(LeNet5, self).__init__()

        self.c1 = C1()
        self.c2_1 = C2()
        self.c2_2 = C2()
        self.c3 = C3()
        self.f4 = F4()
        self.f5 = F5()

    def forward(self, img):
        output = self.c1(img)

        x = self.c2_1(output)
        output = self.c2_2(output)
        
        output += x
        
        output = self.c3(output)
        output = output.view(img.size(0), -1)
        output = self.f4(output)
        output = self.f5(output)
        return output

In [ ]:
x2 = torch.randn(2, 3, 4, 5) # Batch size, Channel, Heigth, Width
x2

In [ ]:
import torch

# Katmanları oluştur
c1 = C1()
c2_1 = C2()
c2_2 = C2()
c3 = C3()

# Input (batch=64)
x = torch.randn(64, 1, 32, 32)

print("Input:", x.shape)

# C1
x = c1(x)
print("After C1:", x.shape)

# C2 (residual yapı)
x1 = c2_1(x)
print("After C2_1:", x1.shape)

x2 = c2_2(x)
print("After C2_2:", x2.shape)

x = x1 + x2
print("After residual sum:", x.shape)

# C3
x = c3(x)
print("After C3:", x.shape)

# Flatten
x = x.view(x.size(0), -1)
print("After flatten:", x.shape)

In [ ]:
x2

## Model Train on Train Dataset

In [ ]:
import torch
import torch.optim as optim
model = LeNet5()
model2 = LeNet5()

In [ ]:
for name, param in model.named_parameters():
    print(f"Parametre adı: {name}")
    print(f"Şekli: {param.shape}")
    print(f"Değerler:\n{param}")
    print("-----------")

In [ ]:
# ===================================================================
# Eğitim Döngüsü: model ve model2 ayrı ayrı eğitilir.
# Epoch: 10, batch_size: 64 (DataLoader'dan), optimizer: Adam(lr=0.001)
# GPU: torch.cuda.is_available() ise "cuda", değilse "cpu"
# ===================================================================
criterion = nn.CrossEntropyLoss()
optimizer  = optim.Adam(model.parameters(),  lr=0.001)
optimizer2 = optim.Adam(model2.parameters(), lr=0.001)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Eğitim cihazı:", device)

model.to(device)
model2.to(device)

EPOCHS = 10

train_losses_1 = []
train_losses_2 = []

# --- Model 1 eğitimi ---
print("\n=== Model 1 (LeNet5) eğitimi ===")
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total   += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc  = 100.0 * correct / total
    train_losses_1.append(epoch_loss)
    print(f"[Model1] Epoch {epoch+1:02d}/{EPOCHS} - loss: {epoch_loss:.4f} - acc: {epoch_acc:.2f}%")

# --- Model 2 eğitimi ---
print("\n=== Model 2 (LeNet5) eğitimi ===")
for epoch in range(EPOCHS):
    model2.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer2.zero_grad()
        outputs = model2(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer2.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total   += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc  = 100.0 * correct / total
    train_losses_2.append(epoch_loss)
    print(f"[Model2] Epoch {epoch+1:02d}/{EPOCHS} - loss: {epoch_loss:.4f} - acc: {epoch_acc:.2f}%")

# Test hücreleri değişmeden çalışabilsin diye modelleri CPU'ya geri taşı
model.to("cpu")
model2.to("cpu")
print("\nEğitim tamamlandı. Modeller CPU'ya alındı; test hücreleri doğrudan çalıştırılabilir.")

## Model Test on Test Dataset

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():  
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Test Accuracy: {accuracy:.2f}%')

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():  
    for images, labels in test_loader:
        outputs = model2(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Test Accuracy: {accuracy:.2f}%')

In [ ]:
correct

## Eğitim Loss Eğrisi Karşılaştırması

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(train_losses_1) + 1), train_losses_1, marker='o', label='Model 1 (LeNet5)')
plt.plot(range(1, len(train_losses_2) + 1), train_losses_2, marker='s', label='Model 2 (LeNet5)')
plt.xlabel('Epoch')
plt.ylabel('Training Loss')
plt.title('İki LeNet5 Modelinin Eğitim Loss Eğrisi')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()